# Safe BO入門：Mixed変数と複数候補（q-batch）

このNotebookでは、**連続変数・整数変数・カテゴリ変数を同時に扱い、複数の実験条件をまとめて提案する方法**を学びます。

まず`44_safe_bo_usage_guide.ipynb`で、GPの学習と`ask()/tell()`の基本操作を確認することをおすすめします。

### この例の入力変数

| 列 | 種類 | 許容値 |
|---|---|---|
| 0 | 連続変数 | 0〜1 |
| 1 | 整数変数 | 0, 1, 2, 3 |
| 2 | カテゴリ変数 | 0, 2, 4 |

**q-batch**とは、一度に`q`個の候補を提案することです。たとえば`q=3`なら、3行×3列の候補テンソルが返ります。

このNotebookではMixed候補生成と安全判定の仕組みに集中するため、**学習済みGPの代わりに予測平均・分散を返す簡易モデル**を使用します。実際にGPを学習する例は前述の基本Notebookを参照してください。robotorchanには`MixedSingleTaskGP`もありますが、この例ではモデル学習を行いません。

**重要：** 安全と「予測」された候補が、実機で必ず安全とは限りません。

In [ ]:
import torch

from robotorchan.optim.domains.variable_space import MixedVariableSpace
from robotorchan.optim.safe_workflow import SafeBOWorkflow
from robotorchan.semantics import ContinuousConstraint, SafetySpecification

torch.set_default_dtype(torch.double)
bounds = torch.tensor([[0.0, 0.0, 0.0], [1.0, 3.0, 4.0]])
space = MixedVariableSpace(
    bounds=bounds,
    integer_dims=(1,),
    categorical_values={2: (0.0, 2.0, 4.0)},
)


def safety_output(X):
    return X[..., :1] - 0.8


class DemoPosterior:
    def __init__(self, X):
        self.mean = safety_output(X)
        self.variance = torch.full_like(self.mean, 0.0001)


class DemoSafetyModel:
    def posterior(self, X):
        return DemoPosterior(X)


class DemoBatchAcquisition:
    def __call__(self, X):
        return X[..., 0].sum(dim=-1)


safety_model = DemoSafetyModel()
acquisition = DemoBatchAcquisition()

## 1. 安全な初期データと3点のバッチ提案

この例では安全出力を`g(X) = X[..., 0] - 0.8`とし、`g(X) <= 0`を安全条件とします。安全性は1列目の連続変数に依存し、他の列は整数・カテゴリの合法値を守る必要があります。

- `initial_X`：安全と確認した最初の観測点
- `SafetySpecification`：安全出力の閾値、初期観測、信頼係数を指定
- `SafeBOWorkflow`：候補の提案（`ask`）と観測値の登録（`tell`）を管理
- `q=3`：3点を同時に提案
- `num_samples=256`：評価する候補バッチ数

`DemoBatchAcquisition`はバッチ内の連続変数値の合計を評価する簡単な関数です。実運用では、q対応の獲得関数を利用します。

候補の形は`[3, 3]`、安全出力の観測値の形は`[3, 1]`です。バッチ内の**全点が予測上安全**であることを確認してから採用します。

In [ ]:
initial_X = torch.tensor([[0.1, 1.0, 2.0]])
specification = SafetySpecification(
    constraints=(ContinuousConstraint(0, 0.0),),
    initial_X=initial_X,
    observed_constraints=safety_output(initial_X),
    beta=4.0,
)
workflow = SafeBOWorkflow(specification, bounds, variable_space=space)

batch = workflow.ask(
    safety_model,
    mode="objective",
    acq_function=acquisition,
    q=3,
    num_samples=256,
    seed=7,
)
print("提案バッチ:", batch)
assert batch.shape == (3, 3)
assert bool((batch[:, 1] == batch[:, 1].round()).all())
assert bool(torch.isin(batch[:, 2], torch.tensor([0.0, 2.0, 4.0])).all())
assert bool((safety_output(batch) + 0.02 <= 0.0).all())

observed_constraints = safety_output(batch)
workflow.tell(batch, observed_constraints)
print("観測登録後の件数:", workflow.observed_X.shape[0])

## 2. 安全領域内の探索（expansion）

`mode="objective"`では獲得関数が高いバッチを選びます。一方、`mode="expansion"`では**安全と予測される領域の中で、安全モデルの不確実性が高い候補**を選びます。

ここでは`q=2`に変更するので、結果は2行×3列です。先ほどの観測と同じく、実測した安全出力を`tell()`で登録します。

注意：この簡易モデルの予測分散は一定なので、探索モードは不確実性の学習効果を示す実験ではありません。SafeOptの厳密な到達可能集合の計算でもありません。

In [ ]:
expansion_batch = workflow.ask(
    safety_model,
    mode="expansion",
    q=2,
    num_samples=256,
    seed=8,
)
print("探索バッチ:", expansion_batch)
assert expansion_batch.shape == (2, 3)
workflow.tell(expansion_batch, safety_output(expansion_batch))
print("観測登録後の件数:", workflow.observed_X.shape[0])

## 3. 学習済みGPを使う場合

このNotebookの`DemoSafetyModel`は、`posterior(X)`から予測平均と分散を返す**説明用のモデル**です。実際のBOでは、robotorchanの`SingleTaskGP`や、カテゴリを含む場合は`MixedSingleTaskGP`など、入力構造に適したGPを学習して置き換えます。

- GP学習：安全出力の観測データでモデルを作成し、`fit()`で学習
- 候補提案：学習済み安全GPとq対応獲得関数を`ask()`に渡す
- 実験：提案されたすべての条件を測定
- 観測登録：安全出力を`tell()`へ渡し、目的関数の観測値も別に保存
- モデル更新：新しい観測を追加してGPと獲得関数を作り直す

**実装上の範囲：** Mixed×q>1は有限個の候補バッチをJoint評価する方式です。候補が見つからない場合は例外となり、危険な候補への自動フォールバックは行いません。実測安全性の保証や非同期バッチ制御は含みません。